<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.1：FIRRTL 介绍

**上一节：[生成器：类型](3.6_types.ipynb)**<br>
**下一节：[FIRRTL AST 遍历](4.2_firrtl_ast_traversal.ipynb)**

## 动机
你已经学习了一些 Scala 并编写了一些 Chisel，对于 90% 的用户来说，这应该足以成为 Chisel 爱好者。

然而，有些用例更适合以 Chisel 设计的程序化变换（transformation）来表达，而不是作为生成器。

例如，假设我们要统计一个设计中的寄存器数量。作为生成器很难做到这一点，因此，我们可以编写一个 FIRRTL Pass 来为我们完成这个任务。

## 环境设置

In [ ]:
val path = System.getProperty("user.dir") + "/../source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.iotesters.{ChiselFlatSpec, Driver, PeekPokeTester}
import firrtl._

## 什么是 FIRRTL？
正如你可能已经注意到的，当你执行一个 Chisel 设计时，它会展开（执行周围的 Scala 代码）以构建你的生成器的一个实例，其中所有 Scala 参数都已解析。

Chisel 不会直接生成 Verilog，而是生成一种称为 FIRRTL 的中间表示形式，它表示已展开（参数已解析）的 RTL 实例。它可以被序列化（转换为字符串以便写入文件），并且这种序列化语法是人类可读的。然而，在内部，它并不是表示为一个长字符串。相反，它是一个组织成节点树的数据结构，称为抽象语法树（AST）。

让我们来看一下！我们将采用一个简单的 Chisel 设计，展开它，并检查它生成了什么 FIRRTL！

首先，我们定义一个 Chisel 模块，它将输入信号延迟两个周期。

In [ ]:
class DelayBy2(width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  val r0 = RegNext(io.in)
  val r1 = RegNext(r0)
  io.out := r1
}

接下来，让我们展开它，序列化，并打印出它生成的 FIRRTL。

In [ ]:
println(chisel3.Driver.emit(() => new DelayBy2(4)))

如你所见，序列化的 FIRRTL 看起来与我们的 Chisel 设计非常相似，所有生成器参数都已解析。

## FIRRTL AST

如前所述，FIRRTL 表示可以序列化为字符串，但在内部，它是一个称为 AST（抽象语法树）的数据结构。这个数据结构是一个节点树，其中一个节点可以包含子节点。这个数据结构中没有循环。

让我们来看看内部数据结构是什么样的：

In [ ]:
val firrtlSerialization = chisel3.Driver.emit(() => new DelayBy2(4))
val firrtlAST = firrtl.Parser.parse(firrtlSerialization.split("\n").toIterator, Parser.GenInfo("file.fir"))

println(firrtlAST)

显然，数据结构的序列化并不那么美观，但你可以看到一些在内部表示 RTL 设计的类等。让我们试着美化一下，以便更容易理解。

In [ ]:
println(stringifyAST(firrtlAST))

这是保存 FIRRTL AST 的内部数据结构。它是一个树形结构，根节点是 **Circuit**，它有 3 个子节点：**@[file.fir@2.0]**、**ArrayBuffer** 和 **cmd5WrapperHelperDelayBy2**。以下是序列化后的 `Circuit` 的实际 Scala 类定义：<a name="circuit"></a><img src="images/circuit.png" alt="Circuit 案例类" />



如你所见，它有三个子节点：`info: Info`、`Modules: Seq[DefModule]` 和 `main: String`。它继承自 `FirrtlNode`，所有 FIRRTL AST 节点都必须这样做。暂时忽略 `def mapXXXX` 函数。

许多 FIRRTL 节点包含一个 `info: Info` 字段，解析器可以在其中插入文件信息，如行号和列号，或者插入一个 `NoInfo` 标记。在此示例中，**@[file.fir@2.0]** 指向 FIRRTL 文件第 2 行第 0 列。

下一节将详细介绍所有这些 FIRRTL 节点。

# FIRRTL 节点描述

本节描述了在 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) 中找到的常见 FirrtlNode。

有关此处未提及的组件的更多详细信息，请参阅 [FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf)。


## Circuit
Circuit 是任何 Firrtl 数据结构的根节点。永远只有一个 Circuit，该 Circuit 包含一个模块定义列表和顶层模块的名称。

#### FirrtlNode 声明
```scala 
Circuit(info: Info, modules: Seq[DefModule], main: String)
```

#### 具体语法
```
circuit Adder:
  ... //模块列表
```
#### 内存表示
```scala
Circuit(NoInfo, Seq(...), "Adder")
```

## Module

模块是 Firrtl 中的模块化单元，从不直接嵌套（声明模块实例有其自己的具体语法和 AST 表示）。每个模块有一个名称、一个端口列表和一个包含其实现的主体。

#### FirrtlNode 声明
```scala
Module(info: Info, name: String, ports: Seq[Port], body: Stmt) extends DefModule
```

#### 具体语法
```
module Adder:
  ... // 端口列表
  ... // 语句
```
#### 内存表示
```scala
Module(NoInfo, "Adder", Seq(...), )
```

## Port
端口定义了模块 IO 的一部分，具有名称、方向（输入或输出）和类型。

#### FirrtlNode 声明
```scala
class Port(info: Info, name: String, direction: Direction, tpe: Type)
```
#### 具体语法
```
input x: UInt
```

#### 内存表示
```scala
Port(NoInfo, "x", INPUT, UIntType(UnknownWidth))
```

## Statement
语句用于描述模块内的组件以及它们如何交互。以下是一些常用的语句：

### 语句块
一组语句。通常用作 Module 声明中的 body 字段。

### 线网声明
线网声明，包含名称和类型。它既可以作为源（从某处连接），也可以作为汇（连接到某处）。
#### FirrtlNode 声明
```scala
DefWire(info: Info, name: String, tpe: Type)
```
#### 具体语法
```
wire w: UInt
```
#### 内存表示
```scala
DefWire(NoInfo, "w", UIntType(UnknownWidth))
```

### 寄存器声明
寄存器声明，包含名称、类型、时钟信号、复位信号和复位值。
#### FirrtlNode 声明
```scala
DefRegister(info: Info, name: String, tpe: Type, clock: Expression, reset: Expression, init: Expression)
```

### 连接
表示从源到汇的有方向连接。注意，它遵循最后一次连接语义，如 Chisel 中所述。

#### FirrtlNode 声明
```scala
Connect(info: Info, loc: Expression, expr: Expression)
```

### 其他语句
其他语句类型如 `DefMemory`、`DefNode`、`IsInvalid`、`Conditionally` 等在此省略；请参阅 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/freechipsproject/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) 了解更多详细信息。

## Expression
表达式表示对已声明组件或逻辑和算术运算的引用。以下是一些常用的表达式：

### Reference
对已声明组件的引用，例如线网、寄存器或端口。它具有名称和类型字段。注意，它不包含指向实际声明的指针，而只包含名称字符串。

#### FirrtlNode 声明
```scala
Reference(name: String, tpe: Type)
```

### DoPrim
一个匿名的原始操作，如 `Add`、`Sub`、`And`、`Or` 或子字选择（`Bits`）。操作的类型由 `op: PrimOp` 字段指示。注意，所需参数和常量的数量由 `op` 决定。

#### FirrtlNode 声明
```scala
DoPrim(op: PrimOp, args: Seq[Expression], consts: Seq[BigInt], tpe: Type)
```

### 其他表达式
其他表达式包括 `SubField`、`SubIndex`、`SubAccess`、`Mux`、`ValidIf` 等，在 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) 和 [FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf) 中有更详细的描述。

# 回到我们的示例

让我们再次看一下我们示例中的 FIRRTL AST。希望设计结构现在更清晰了！

In [ ]:
println(stringifyAST(firrtlAST))

本节到此结束！在下一节中，我们将学习 FIRRTL 变换（transformation）如何遍历这个 AST 并修改它。